# FieldAid: Offline Disaster Response Copilot
## Gemma 4 Good Hackathon Demo

This notebook bootstraps FieldAid from scratch and demonstrates the complete offline disaster-response workflow powered by Gemma 4 (E4B) running locally via Ollama.

Demo capabilities include:
- Shelter triage
- Damage assessment
- Trust and grounding verification
- Incident dashboard
- Discord handoff drafting
- Offline sync exports
- Full FastAPI server walkthrough

---
## Part 1: Clone the Repository

In [ ]:
!git clone https://github.com/Kushalk0677/fieldaid-offline-disaster-copilot.git --depth 1

%cd fieldaid-offline-disaster-copilot

print("Repository cloned successfully.")

---
## Part 2: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

import sys

print(f"Python version: {sys.version}")

from app.main import app
print("FastAPI app imported successfully.")

---
## Part 3: Install and Start Ollama

This step installs Ollama, launches the Ollama runtime, and pulls the `gemma4:e4b` model.

Expected model size: ~9.6 GB
Estimated download time: 5–10 minutes depending on connection speed.

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd curl > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

import os
import subprocess
import time

os.system("killall -9 ollama 2>/dev/null || true")

print("Starting Ollama server...")
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(3)

print("Pulling gemma4:e4b model (~9.6 GB)...")
!ollama pull gemma4:e4b

print("\nVerifying model response...")
!ollama run gemma4:e4b "Say hello in one word." --quiet

In [ ]:
!ps aux | grep -i ollama | grep -v grep

print("\nInstalled Ollama models:")
!ollama list

---
## Part 4: Configure Runtime Environment

In [ ]:
import os
from pathlib import Path

os.environ["FIELDAID_RUNTIME_CACHE"] = str(Path.home() / ".fieldaid_runtime")
os.environ["PYTHONPATH"] = "."

Path("data/uploads").mkdir(parents=True, exist_ok=True)
Path(os.environ["FIELDAID_RUNTIME_CACHE"]).mkdir(parents=True, exist_ok=True)

print("Runtime configured successfully.\n")

print(f"Upload directory: {Path('data/uploads').absolute()}")
print(f"Runtime cache: {os.environ['FIELDAID_RUNTIME_CACHE']}")
print(f"Database path: {Path('data/fieldaid.sqlite').absolute()}")

classifier = Path("models/fieldaid-medic-image-classifier-final/best.pt")
yolo = Path("yolo11n.pt")

for name, path in [("Classifier", classifier), ("YOLO", yolo)]:
    if path.exists():
        print(f"{name}: found ({path.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"{name}: missing")

---
## Part 5: Start the FieldAid Web Server

In [ ]:
import os
import threading
import time

def run_server():
    os.system(
        "uvicorn app.main:app --host 0.0.0.0 --port 8000 > /tmp/fieldaid.log 2>&1"
    )

print("Starting FieldAid server on port 8000...")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(4)

import httpx

try:
    resp = httpx.get("http://localhost:8000", timeout=5)
    print(f"Server running successfully (status {resp.status_code})")

    try:
        from google.colab import output

        output.serve_kernel_port_as_window(8000)
        print("Colab opened the FieldAid server window.")

    except Exception:
        print("Open http://localhost:8000 if running outside Colab.")

    print("Click the 'Start Walkthrough' button in the bottom-right corner once the UI loads.")

except Exception as e:
    print(f"Server not ready yet: {e}")
    print("Run this cell again after a few seconds.")

---
## Part 6: Demo API Flows

### 6A: Shelter Intake Analysis

In [ ]:
from fastapi.testclient import TestClient
from app.main import app
import json

client = TestClient(app)

response = client.post(
    "/api/analyze",
    data={
        "scenario_type": "shelter",
        "note_text": "43 people in the school building. 6 elderly residents, 2 insulin patients need refrigeration. Water left for about 8 hours. Bridge to main road is blocked.",
        "location": "Government School, Sector 7",
        "model": "gemma4:e4b",
    },
)

if response.status_code == 200:
    print(json.dumps(response.json(), indent=2))
else:
    print(f"Error {response.status_code}: {response.text}")

### 6B: Trust and Grounding Verification

In [ ]:
response = client.post(
    "/api/grounding",
    data={
        "question": "Can civilians safely cross the damaged Creek Road bridge now?",
        "scenario_type": "damage",
        "model": "gemma4:e4b",
    },
)

if response.status_code == 200:
    print(json.dumps(response.json(), indent=2))
else:
    print(f"Error {response.status_code}: {response.text}")

### 6C: Dashboard and Incident Feed

In [ ]:
dashboard = client.get("/api/dashboard")
incidents = client.get("/api/incidents")

if dashboard.status_code == 200:
    print("Dashboard Data:\n")
    print(json.dumps(dashboard.json(), indent=2))

if incidents.status_code == 200:
    print(f"\nTotal incidents recorded: {len(incidents.json())}")

### 6D: Discord Handoff Draft

In [ ]:
handoff = client.post(
    "/api/outbox",
    json={
        "incident_id": 1,
        "channel": "discord",
        "destination": "field-response-team",
        "message": "URGENT: 43 people sheltered. Insulin patients need refrigeration. Water running out. Bridge blocked.",
    },
)

outbox = client.get("/api/outbox")

print("Drafted handoff message:\n")
print(json.dumps(handoff.json(), indent=2))

print(f"\nOutbox messages queued: {len(outbox.json())}")

### 6E: Offline Sync Export

In [ ]:
export_data = client.get("/api/sync/export")

if export_data.status_code == 200:
    print(json.dumps(export_data.json(), indent=2))

sync_queue = client.get("/api/sync/queue")

if sync_queue.status_code == 200:
    print(f"\nPending sync queue items: {len(sync_queue.json())}")

---
## Part 7: Run Test Suite

In [ ]:
!python -m pytest tests/ -v --tb=short

print("\nTest suite execution complete.")